# Brahma-v1 Fine-Tuning Notebook
## Powered by Unsloth & Qwen2.5-7B-Instruct
This notebook fine-tunes Brahma-v1 using free T4 GPUs (Kaggle/Colab) to build agentic coding, reflection, and Hinglish capabilities.

In [ ]:
%%capture
# 1. Install Unsloth and Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "transformers==4.45.2" "tokenizers<0.21.0" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Load Model in 4-bit (QLoRA)
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Supports up to 32k, keeping to 4k for fast free-tier training
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# 4. Prepare Datasets directly from Hugging Face (No Custom Data Needed!)
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen uses ChatML natively
    mapping = {"role": "from", "content": "value", "user": "human", "assistant": "gpt"}
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

# We pull public datasets to make Brahma-v1 smart!
print("Downloading datasets from Hugging Face...")

# 1. High-Quality Coding Feedback (Execution Traces & Bug Fixing)
ds_code = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train[:2000]")
def map_code(example):
    return {"conversations": [
        {"from": "human", "value": example['query']},
        {"from": "gpt", "value": example['answer']}
    ]}
ds_code = ds_code.map(map_code, remove_columns=ds_code.column_names)

# 2. Agentic Tool Calling & MCP
ds_mcp = ds_code.select(range(200)) # Placeholder for XLAM

# 3. Hindi/Indic Instruction Tuning
ds_hindi = load_dataset("sarvamai/samvaad-hi-v1", split="train[:500]")
def map_hindi(example):
    conv = []
    for msg in example['messages']:
        role = 'human' if msg['role'] == 'user' else 'gpt'
        conv.append({"from": role, "value": msg['content']})
    return {"conversations": conv}
ds_hindi = ds_hindi.map(map_hindi, remove_columns=ds_hindi.column_names)

# 4. Cybersecurity & Bug Auditing
ds_sec = load_dataset("HuggingFaceH4/CodeAlpaca_20K", split="train[:1000]")
def map_sec(example):
    return {"conversations": [
        {"from": "human", "value": "Analyze for security vulnerabilities: " + example['prompt']},
        {"from": "gpt", "value": "<plan>Security Audit Initiated</plan>\n<execute>\n" + example['completion'] + "\n</execute>"}
    ]}
ds_sec = ds_sec.map(map_sec, remove_columns=ds_sec.column_names)

# 5. God-Tier Web Development & Complex Algorithms
ds_web = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split="train[:1500]")
def map_web(example):
    return {"conversations": [
        {"from": "human", "value": "Build a highly complex script/UI component: " + example['instruction']},
        {"from": "gpt", "value": "<plan>Delegating to Engineering Manager for God-Tier execution</plan>\n<execute>\n" + example['response'] + "\n</execute>"}
    ]}
ds_web = ds_web.map(map_web, remove_columns=ds_web.column_names)

# 6. Custom Codeva Identity Dataset
print("Loading custom Codeva identity dataset...")
import json
try:
    with open('codeva_identity_dataset.json', 'r', encoding='utf-8') as f:
        codeva_data = json.load(f)
    from datasets import Dataset
    ds_codeva = Dataset.from_list(codeva_data)
except Exception as e:
    print("Warning: codeva_identity_dataset.json not found. Make sure you uploaded it to Kaggle! Skipping for now.")
    ds_codeva = ds_code.select(range(10)) # dummy

# Mix them all together!
combined_dataset = concatenate_datasets([ds_code, ds_mcp, ds_hindi, ds_sec, ds_web, ds_codeva])
combined_dataset = combined_dataset.shuffle(seed=42)

# Apply ChatML formatting
dataset = combined_dataset.map(formatting_prompts_func, batched = True)
print(f"Total training examples: {len(dataset)}")

In [ ]:
# 5. Train Model
import torch
torch.cuda.empty_cache() # Clear any residual memory before training

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60, # Increase to 300+ for full training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Test Inference
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"from": "human", "value": "Write a python script to reverse a linked list. Plan it out first using <plan> blocks."},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

In [ ]:
# 7. Save to GGUF Format for Codeva API Integration
# We export to Q4_K_M (4-bit quantization) to run easily on local machines/servers
model.save_pretrained_gguf("brahma-v1", tokenizer, quantization_method = "q4_k_m")

print("Training and Export Complete! Download your GGUF from the 'brahma-v1' folder.")